In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load development dataset
data_path = Path("C:/Users/zeina/data/processed/dev_df.parquet")

dev_df = pd.read_parquet(data_path)

target_col = "isFraud"
id_col = "TransactionID"
time_col = "TransactionDT"

print("Dataset shape:", dev_df.shape)

Dataset shape: (472432, 434)


# 1- Leakage reasoning

In [3]:
# ---1.1.Define core structural columns----------------------------------------

target_col = "isFraud"
id_col = "TransactionID"
time_col = "TransactionDT"


# ---1.2.Assign initial feature roles------------------------------------------

column_roles = pd.DataFrame({
    "column_name": dev_df.columns
})

column_roles["role"] = "predictor_candidate"
column_roles.loc[column_roles["column_name"] == target_col, "role"] = "target"
column_roles.loc[column_roles["column_name"] == id_col, "role"] = "identifier"
column_roles.loc[column_roles["column_name"] == time_col, "role"] = "time_index"


# ---1.3.Build uniqueness summary----------------------------------------------

# High uniqueness can indicate identifier-like behavior
uniqueness_summary = (
    dev_df.nunique(dropna=False)
    .to_frame(name="n_unique")
    .assign(
        unique_ratio=lambda df_: df_["n_unique"] / len(dev_df)
    )
    .sort_values("unique_ratio", ascending=False)
    .reset_index()
    .rename(columns={"index": "column_name"})
)

display(uniqueness_summary.head(20))


# ---1.4.Flag identifier-like columns for review-------------------------------

# These columns are not automatically dropped unless the reason is clear
identifier_like_review = (
    uniqueness_summary.loc[
        ~uniqueness_summary["column_name"].isin([target_col, id_col]) &
        (uniqueness_summary["unique_ratio"] >= 0.95)
    ]
    .copy()
)

display(identifier_like_review)


# ---1.5.Create direct model exclusion table-----------------------------------

# Exclude only columns with a clear and defensible reason
direct_model_exclusions = pd.DataFrame({
    "column_name": [id_col, time_col],
    "exclude_from_direct_modeling": [True, True],
    "allowed_for_feature_engineering": [False, True],
    "reason": [
        "Pure row identifier. It does not represent transaction behavior and can encourage memorization.",
        "Raw time index can leak temporal position. Keep for time-based feature engineering, not as a raw model input."
    ]
})

display(direct_model_exclusions)


# ---1.6.Create final exclusion lists------------------------------------------

drop_from_direct_modeling = direct_model_exclusions.loc[
    direct_model_exclusions["exclude_from_direct_modeling"],
    "column_name"
].tolist()

feature_engineering_only = direct_model_exclusions.loc[
    direct_model_exclusions["allowed_for_feature_engineering"],
    "column_name"
].tolist()

print("Drop from direct modeling:", drop_from_direct_modeling)
print("Allowed for feature engineering only:", feature_engineering_only)


# ---1.7.Create leakage reasoning summary--------------------------------------

leakage_summary = pd.DataFrame({
    "item": [
        "Target column",
        "Identifier column",
        "Raw time column",
        "High-uniqueness columns flagged for review"
    ],
    "decision": [
        "Keep as target only",
        "Exclude from modeling",
        "Do not model directly",
        "Review before modeling"
    ],
    "notes": [
        "Used only as the response variable.",
        "Used only for joins, split alignment, and audit checks.",
        "Use only to derive safer time features later.",
        "High uniqueness can indicate identifier-like behavior, but requires column-specific judgment."
    ]
})

display(leakage_summary)

,column_name,n_unique,unique_ratio
0,TransactionID,472432,1.000000
1,TransactionDT,457939,0.969323
2,id_02,99488,0.210587
3,V307,30772,0.065135
4,V127,19880,0.042080
5,V308,18381,0.038907
6,TransactionAmt,18131,0.038378
7,V310,16698,0.035345
8,card1,12730,0.026946
9,V306,12611,0.026694


,column_name,n_unique,unique_ratio
1,TransactionDT,457939,0.969323


,column_name,exclude_from_direct_modeling,allowed_for_feature_engineering,reason
0,TransactionID,True,False,Pure row identifier. It does not represent tra...
1,TransactionDT,True,True,Raw time index can leak temporal position. Kee...


Drop from direct modeling: ['TransactionID', 'TransactionDT']
Allowed for feature engineering only: ['TransactionDT']


,item,decision,notes
0,Target column,Keep as target only,Used only as the response variable.
1,Identifier column,Exclude from modeling,"Used only for joins, split alignment, and audi..."
2,Raw time column,Do not model directly,Use only to derive safer time features later.
3,High-uniqueness columns flagged for review,Review before modeling,High uniqueness can indicate identifier-like b...


# 2- Feature engineering plan

In [4]:
# ---2.1.Feature engineering roadmap-------------------------------------------

# Feature engineering in this project should stay conservative and explainable.
# We only create features that are defensible from EDA and safe from leakage.

feature_families = {
    "transaction_amount": {
        "source_columns": ["TransactionAmt"],
        "planned_features": [
            "TransactionAmt_log1p",
            "TransactionAmt_is_small",
            "TransactionAmt_is_large",
        ],
        "reason": (
            "TransactionAmt is heavily right-skewed, and EDA showed a non-linear "
            "relationship with fraud risk."
        ),
    },
    "transaction_time": {
        "source_columns": ["TransactionDT"],
        "planned_features": [
            "TransactionHour",
            "TransactionDay",
            "TransactionWeek",
        ],
        "reason": (
            "Raw TransactionDT should not be modeled directly, but safer derived "
            "time features may capture temporal behavior and drift."
        ),
    },
    "missingness": {
        "source_columns": [
            "selected high-value fields with meaningful missingness"
        ],
        "planned_features": [
            "missing_indicators_for_selected_columns"
        ],
        "reason": (
            "Missingness is widespread and likely structural. In fraud settings, "
            "missingness itself can carry signal."
        ),
    },
    "email_domains": {
        "source_columns": ["P_emaildomain", "R_emaildomain"],
        "planned_features": [
            "email_domain_provider_group",
            "email_domain_suffix_group",
            "email_domain_match_flag",
        ],
        "reason": (
            "EDA showed meaningful fraud differences across purchaser and recipient "
            "email domains."
        ),
    },
    "device_signals": {
        "source_columns": ["DeviceType", "DeviceInfo"],
        "planned_features": [
            "DeviceInfo_simplified",
            "is_mobile_device",
        ],
        "reason": (
            "Device-related fields showed fraud separation, but raw values may be "
            "too noisy or high-cardinality."
        ),
    },
}

print("Planned feature families:")
for family_name, family_info in feature_families.items():
    print(f"\n{family_name}")
    print(f"  source columns : {family_info['source_columns']}")
    print(f"  planned features: {family_info['planned_features']}")
    print(f"  reason         : {family_info['reason']}")


# ---2.2.First implementation priority-----------------------------------------

first_feature_family = "transaction_amount"

print("\nFirst feature family to implement:", first_feature_family)
print(
    "Reason: it is simple, explainable, supported by EDA, and does not require "
    "complex joins or high-cardinality handling."
)

Planned feature families:

transaction_amount
  source columns : ['TransactionAmt']
  planned features: ['TransactionAmt_log1p', 'TransactionAmt_is_small', 'TransactionAmt_is_large']
  reason         : TransactionAmt is heavily right-skewed, and EDA showed a non-linear relationship with fraud risk.

transaction_time
  source columns : ['TransactionDT']
  planned features: ['TransactionHour', 'TransactionDay', 'TransactionWeek']
  reason         : Raw TransactionDT should not be modeled directly, but safer derived time features may capture temporal behavior and drift.

missingness
  source columns : ['selected high-value fields with meaningful missingness']
  planned features: ['missing_indicators_for_selected_columns']
  reason         : Missingness is widespread and likely structural. In fraud settings, missingness itself can carry signal.

email_domains
  source columns : ['P_emaildomain', 'R_emaildomain']
  planned features: ['email_domain_provider_group', 'email_domain_suffix_group

### Interpretation:

- `TransactionID` is a pure row identifier and contains no predictive signal. It will be excluded from modeling.
- `TransactionDT` has very high uniqueness and should not be used directly as a feature.
- However, `TransactionDT` can still be used to derive safer time-based features (hour, day, etc.).
- High-cardinality fields such as `id_02` require careful review but should not be dropped automatically.

# 3.Build candidate engineered features

In [6]:
# ---3.1.Working copy----------------------------------------------------------

# Work on a copy to avoid modifying the original dataset during experimentation
df = dev_df.copy()


# ---3.2.Transaction amount transformations-----------------------------------

# Log transform helps stabilize heavy right skew in transaction amounts
df["TransactionAmt_log1p"] = np.log1p(df["TransactionAmt"])

# Flag extremely small transactions (often associated with card testing)
df["TransactionAmt_is_small"] = (df["TransactionAmt"] < 5).astype(int)

# Flag very large transactions (large purchases can also carry higher fraud risk)
df["TransactionAmt_is_large"] = (df["TransactionAmt"] > 500).astype(int)


# ---3.3.Attach engineered features to main dataset---------------------------

dev_df["TransactionAmt_log1p"] = df["TransactionAmt_log1p"]
dev_df["TransactionAmt_is_small"] = df["TransactionAmt_is_small"]
dev_df["TransactionAmt_is_large"] = df["TransactionAmt_is_large"]


# ---3.4.Basic validation------------------------------------------------------

new_features = [
    "TransactionAmt_log1p",
    "TransactionAmt_is_small",
    "TransactionAmt_is_large"
]

print("New features added:")
print(new_features)


print("\nPreview of engineered features:")
dev_df[
    [
        "TransactionAmt",
        "TransactionAmt_log1p",
        "TransactionAmt_is_small",
        "TransactionAmt_is_large"
    ]
].head()


# ---3.5.Fraud rate check for engineered flags--------------------------------

fraud_rate_small = dev_df.groupby("TransactionAmt_is_small")["isFraud"].mean()
fraud_rate_large = dev_df.groupby("TransactionAmt_is_large")["isFraud"].mean()

print("\nFraud rate by small transaction flag:")
print(fraud_rate_small)

print("\nFraud rate by large transaction flag:")
print(fraud_rate_large)

New features added:
['TransactionAmt_log1p', 'TransactionAmt_is_small', 'TransactionAmt_is_large']

Preview of engineered features:

Fraud rate by small transaction flag:
TransactionAmt_is_small
0    0.035077
1    0.065217
Name: isFraud, dtype: float64

Fraud rate by large transaction flag:
TransactionAmt_is_large
0    0.034829
1    0.042850
Name: isFraud, dtype: float64


# 4.Time-based feature engineering

In [7]:
# ---4.1.Derive safe time features from TransactionDT--------------------------

# TransactionDT is a relative time index in seconds from a reference point.
# We do not use it directly, but we can derive simpler calendar-like features.

seconds_in_day = 24 * 60 * 60
seconds_in_week = 7 * seconds_in_day

# Hour of day: 0 to 23
dev_df["TransactionHour"] = (dev_df["TransactionDT"] // 3600) % 24

# Day index since the reference start
dev_df["TransactionDay"] = dev_df["TransactionDT"] // seconds_in_day

# Week index since the reference start
dev_df["TransactionWeek"] = dev_df["TransactionDT"] // seconds_in_week


# ---4.2.Basic validation------------------------------------------------------

time_feature_cols = [
    "TransactionDT",
    "TransactionHour",
    "TransactionDay",
    "TransactionWeek",
]

print("New time-based features added:")
print(time_feature_cols[1:])

print("\nPreview of time-based features:")
display(dev_df[time_feature_cols].head())


# ---4.3.Range checks----------------------------------------------------------

print("\nTransactionHour range:")
print(dev_df["TransactionHour"].min(), "to", dev_df["TransactionHour"].max())

print("\nTransactionDay range:")
print(dev_df["TransactionDay"].min(), "to", dev_df["TransactionDay"].max())

print("\nTransactionWeek range:")
print(dev_df["TransactionWeek"].min(), "to", dev_df["TransactionWeek"].max())

New time-based features added:
['TransactionHour', 'TransactionDay', 'TransactionWeek']

Preview of time-based features:


,TransactionDT,TransactionHour,TransactionDay,TransactionWeek
0,86400,0,1,0
1,86401,0,1,0
2,86469,0,1,0
3,86499,0,1,0
4,86506,0,1,0



TransactionHour range:
0 to 23

TransactionDay range:
1 to 141

TransactionWeek range:
0 to 20


### Interpretation

- TransactionDT is measured in seconds from a reference point.
- Converting it into hour, day, and week creates interpretable behavioral timing features.
- The ranges look reasonable:
  - Hour: 0–23
  - Day: 1–141
  - Week: 0–20
- These derived features allow the model to capture time-based fraud patterns without using the raw timestamp directly.

In [8]:
# ---4.4.Fraud rate by transaction hour---------------------------------------

fraud_by_hour = (
    dev_df
    .groupby("TransactionHour")["isFraud"]
    .mean()
    .sort_index()
)

print("Fraud rate by transaction hour:")
print(fraud_by_hour)

Fraud rate by transaction hour:
TransactionHour
0     0.032038
1     0.032198
2     0.036993
3     0.036078
4     0.049591
5     0.066611
6     0.073840
7     0.101077
8     0.097203
9     0.105748
10    0.069388
11    0.039462
12    0.030001
13    0.021433
14    0.024500
15    0.024985
16    0.028285
17    0.032678
18    0.035033
19    0.034929
20    0.034463
21    0.033678
22    0.032162
23    0.038892
Name: isFraud, dtype: float64


### Interpretation
Fraud rates vary substantially across hours of the day

Early morning hours (approximately 5–9) show significantly higher fraud rates, reaching over 10% in some periods, compared to the overall baseline of about 3.%.

This suggests that transaction timing contains meaningful behavioral signal and supports including `TransactionHour` as a model feature.

In [9]:
# ---4.5.Cyclical encoding of transaction hour--------------------------------

# Convert hour into cyclical representation
# This captures the circular nature of time (23 -> 0 transition)

dev_df["TransactionHour_sin"] = np.sin(2 * np.pi * dev_df["TransactionHour"] / 24)
dev_df["TransactionHour_cos"] = np.cos(2 * np.pi * dev_df["TransactionHour"] / 24)


print("Cyclical hour features added:")
print(["TransactionHour_sin", "TransactionHour_cos"])

print("\nPreview:")
dev_df[
    [
        "TransactionHour",
        "TransactionHour_sin",
        "TransactionHour_cos"
    ]
].head()

Cyclical hour features added:
['TransactionHour_sin', 'TransactionHour_cos']

Preview:


,TransactionHour,TransactionHour_sin,TransactionHour_cos
0,0,0.0,1.0
1,0,0.0,1.0
2,0,0.0,1.0
3,0,0.0,1.0
4,0,0.0,1.0


In [10]:
# ---4.6.Cyclical encoding sanity check---------------------------------------

sample_hours = dev_df[
    ["TransactionHour", "TransactionHour_sin", "TransactionHour_cos"]
].drop_duplicates().sort_values("TransactionHour")

print(sample_hours.head(10))

      TransactionHour  TransactionHour_sin  TransactionHour_cos
0                   0             0.000000         1.000000e+00
228                 1             0.258819         9.659258e-01
429                 2             0.500000         8.660254e-01
609                 3             0.707107         7.071068e-01
743                 4             0.866025         5.000000e-01
835                 5             0.965926         2.588190e-01
899                 6             1.000000         6.123234e-17
990                 7             0.965926        -2.588190e-01
1056                8             0.866025        -5.000000e-01
1094                9             0.707107        -7.071068e-01


# 5.Missingness feature engineering

In [11]:
# ---5.1.Identify columns with substantial missingness-------------------------

missing_pct = dev_df.isnull().mean()

missing_summary = (
    missing_pct
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["feature", "missing_pct"]

print("Top features by missing percentage:")
print(missing_summary.head(20))


# Select candidate columns where missingness is high but not extreme
# (very close to 100% missing would not be useful)

candidate_missing_features = missing_summary[
    (missing_summary["missing_pct"] > 0.30) &
    (missing_summary["missing_pct"] < 0.95)
]["feature"].tolist()

print("\nNumber of candidate features for missing indicators:")
print(len(candidate_missing_features))

print("\nSample candidate features:")
print(candidate_missing_features[:15])

Top features by missing percentage:
   feature  missing_pct
0    id_24     0.991715
1    id_25     0.991012
2    id_08     0.990981
3    id_07     0.990981
4    id_21     0.990976
5    id_26     0.990968
6    id_27     0.990955
7    id_23     0.990955
8    id_22     0.990955
9       D7     0.936617
10   dist2     0.932054
11   id_18     0.920789
12     D13     0.896339
13     D14     0.894512
14     D12     0.888443
15   id_04     0.885507
16   id_03     0.885507
17      D6     0.876209
18   id_10     0.869590
19      D9     0.869590

Number of candidate features for missing indicators:
223

Sample candidate features:
['D7', 'dist2', 'id_18', 'D13', 'D14', 'D12', 'id_04', 'id_03', 'D6', 'id_10', 'D9', 'D8', 'id_09', 'id_33', 'id_30']


In [12]:
# ---5.2.Measure fraud signal of missingness-----------------------------------

missing_signal = []

for col in candidate_missing_features:
    
    missing_flag = dev_df[col].isnull().astype(int)
    
    fraud_rate_missing = dev_df.loc[missing_flag == 1, "isFraud"].mean()
    fraud_rate_present = dev_df.loc[missing_flag == 0, "isFraud"].mean()
    
    signal_strength = abs(fraud_rate_missing - fraud_rate_present)
    
    missing_signal.append({
        "feature": col,
        "missing_pct": dev_df[col].isnull().mean(),
        "fraud_missing": fraud_rate_missing,
        "fraud_present": fraud_rate_present,
        "fraud_gap": signal_strength
    })

missing_signal_df = pd.DataFrame(missing_signal)

missing_signal_df = missing_signal_df.sort_values(
    "fraud_gap",
    ascending=False
)

print("Top missingness signals:")
print(missing_signal_df.head(20))

Top missingness signals:
           feature  missing_pct  fraud_missing  fraud_present  fraud_gap
0               D7     0.936617       0.027384       0.149679   0.122296
4              D14     0.894512       0.025762       0.114616   0.088854
5              D12     0.888443       0.025269       0.113713   0.088444
3              D13     0.896339       0.026642       0.108570   0.081928
6            id_04     0.885507       0.026184       0.104363   0.078179
7            id_03     0.885507       0.026184       0.104363   0.078179
8               D6     0.876209       0.025530       0.103124   0.077594
12           id_09     0.869590       0.025198       0.101396   0.076198
9            id_10     0.869590       0.025198       0.101396   0.076198
10              D9     0.869590       0.025198       0.101396   0.076198
11              D8     0.869590       0.025198       0.101396   0.076198
1            dist2     0.932054       0.030786       0.094798   0.064012
66           id_13     0.7

In [13]:
# ---5.3.Select strongest missingness signals----------------------------------

top_missing_features = (
    missing_signal_df
    .head(15)["feature"]
    .tolist()
)

print("Selected features for missing indicators:")
print(top_missing_features)

Selected features for missing indicators:
['D7', 'D14', 'D12', 'D13', 'id_04', 'id_03', 'D6', 'id_09', 'id_10', 'D9', 'D8', 'dist2', 'id_13', 'R_emaildomain', 'id_05']


In [14]:
# ---5.4.Create missing indicators---------------------------------------------

for col in top_missing_features:
    
    indicator_name = f"{col}_missing"
    
    dev_df[indicator_name] = dev_df[col].isnull().astype(int)

print("Missing indicators created:")
print([f"{col}_missing" for col in top_missing_features])

Missing indicators created:
['D7_missing', 'D14_missing', 'D12_missing', 'D13_missing', 'id_04_missing', 'id_03_missing', 'D6_missing', 'id_09_missing', 'id_10_missing', 'D9_missing', 'D8_missing', 'dist2_missing', 'id_13_missing', 'R_emaildomain_missing', 'id_05_missing']


In [15]:
# ---5.5.Verify missing indicators--------------------------------------------

indicator_cols = [f"{col}_missing" for col in top_missing_features]

print("Preview of missing indicators:")

dev_df[
    top_missing_features[:3] + indicator_cols[:3]
].head()

Preview of missing indicators:


,D7,D14,D12,D7_missing,D14_missing,D12_missing
0,NaN,NaN,NaN,1,1,1
1,NaN,NaN,NaN,1,1,1
2,NaN,NaN,NaN,1,1,1
3,NaN,NaN,NaN,1,1,1
4,NaN,NaN,NaN,1,1,1


In [16]:
dev_df[
    top_missing_features[:3] + indicator_cols[:3]
].dropna().head()

,D7,D14,D12,D7_missing,D14_missing,D12_missing
10,0.0,0.0,0.0,0,0,0
40,0.0,0.0,0.0,0,0,0
130,0.0,0.0,0.0,0,0,0
199,48.0,0.0,398.0,0,0,0
209,24.0,0.0,24.0,0,0,0


# 6.Email domain feature engineering

In [18]:
# ---6.1.Extract raw email provider and suffix---------------------------------

# Split email domains into provider and suffix components
# Example: gmail.com -> provider=gmail, suffix=com

dev_df["P_email_provider"] = dev_df["P_emaildomain"].str.split(".").str[0]
dev_df["R_email_provider"] = dev_df["R_emaildomain"].str.split(".").str[0]

dev_df["P_email_suffix"] = dev_df["P_emaildomain"].str.split(".").str[-1]
dev_df["R_email_suffix"] = dev_df["R_emaildomain"].str.split(".").str[-1]


# ---6.2.Create purchaser/recipient domain match flags-------------------------

# Exact match at full domain level
dev_df["email_domain_match"] = (
    dev_df["P_emaildomain"] == dev_df["R_emaildomain"]
).astype(int)

# Match at provider level only
dev_df["email_provider_match"] = (
    dev_df["P_email_provider"] == dev_df["R_email_provider"]
).astype(int)

# Match at suffix level only
dev_df["email_suffix_match"] = (
    dev_df["P_email_suffix"] == dev_df["R_email_suffix"]
).astype(int)


# ---6.3.Create missingness-aware email availability flags---------------------

# These flags separate "both missing" from "present but different"
dev_df["P_emaildomain_missing"] = dev_df["P_emaildomain"].isnull().astype(int)
dev_df["R_emaildomain_missing"] = dev_df["R_emaildomain"].isnull().astype(int)

dev_df["both_email_missing"] = (
    dev_df["P_emaildomain"].isnull() & dev_df["R_emaildomain"].isnull()
).astype(int)

dev_df["both_email_present"] = (
    dev_df["P_emaildomain"].notnull() & dev_df["R_emaildomain"].notnull()
).astype(int)


# ---6.4.Basic validation------------------------------------------------------

email_feature_cols = [
    "P_email_provider",
    "R_email_provider",
    "P_email_suffix",
    "R_email_suffix",
    "email_domain_match",
    "email_provider_match",
    "email_suffix_match",
    "P_emaildomain_missing",
    "R_emaildomain_missing",
    "both_email_missing",
    "both_email_present",
]

print("Email domain features added:")
print(email_feature_cols)

print("\nPreview of email domain features:")
display(
    dev_df[
        [
            "P_emaildomain",
            "R_emaildomain",
            "P_email_provider",
            "R_email_provider",
            "P_email_suffix",
            "R_email_suffix",
            "email_domain_match",
            "email_provider_match",
            "email_suffix_match",
            "both_email_missing",
            "both_email_present",
        ]
    ].head(10)
)

Email domain features added:
['P_email_provider', 'R_email_provider', 'P_email_suffix', 'R_email_suffix', 'email_domain_match', 'email_provider_match', 'email_suffix_match', 'P_emaildomain_missing', 'R_emaildomain_missing', 'both_email_missing', 'both_email_present']

Preview of email domain features:


,P_emaildomain,R_emaildomain,P_email_provider,R_email_provider,P_email_suffix,R_email_suffix,email_domain_match,email_provider_match,email_suffix_match,both_email_missing,both_email_present
0,None,None,None,None,None,None,0,0,0,1,0
1,gmail.com,None,gmail,None,com,None,0,0,0,0,0
2,outlook.com,None,outlook,None,com,None,0,0,0,0,0
3,yahoo.com,None,yahoo,None,com,None,0,0,0,0,0
4,gmail.com,None,gmail,None,com,None,0,0,0,0,0
5,gmail.com,None,gmail,None,com,None,0,0,0,0,0
6,yahoo.com,None,yahoo,None,com,None,0,0,0,0,0
7,mail.com,None,mail,None,com,None,0,0,0,0,0
8,anonymous.com,None,anonymous,None,com,None,0,0,0,0,0
9,yahoo.com,None,yahoo,None,com,None,0,0,0,0,0


### Interpretation

Email domain fields showed meaningful fraud differences during EDA, so we created structured features from them.

Provider and suffix components capture reusable patterns across domains (for example gmail, yahoo, outlook), while match flags measure whether purchaser and recipient email attributes align.

Additional availability indicators distinguish between cases where email metadata is present versus missing, which may also carry fraud signal.

# 7.Feature signal checks

In [19]:
# ---7.1.Email feature signal--------------------------------------------------

email_signal_cols = [
    "email_domain_match",
    "email_provider_match",
    "email_suffix_match",
    "both_email_missing",
    "both_email_present"
]

for col in email_signal_cols:

    fraud_rates = dev_df.groupby(col)["isFraud"].mean()

    print(f"\nFraud rate by {col}:")
    print(fraud_rates)


Fraud rate by email_domain_match:
email_domain_match
0    0.022313
1    0.092873
Name: isFraud, dtype: float64

Fraud rate by email_provider_match:
email_provider_match
0    0.022314
1    0.092857
Name: isFraud, dtype: float64

Fraud rate by email_suffix_match:
email_suffix_match
0    0.021895
1    0.083123
Name: isFraud, dtype: float64

Fraud rate by both_email_missing:
both_email_missing
0    0.036598
1    0.026117
Name: isFraud, dtype: float64

Fraud rate by both_email_present:
both_email_present
0    0.021991
1    0.080590
Name: isFraud, dtype: float64


### Interpretation

Email-related features show strong fraud separation.

Transactions where purchaser and recipient email domains match have substantially higher fraud rates (around 9%) compared with non-matching domains (around 2%). Similar patterns appear for provider and suffix matches.

This suggests that relationships between purchaser and recipient email metadata capture useful behavioral signals.

Email availability also carries signal: transactions where both email fields are present show higher fraud prevalence than those where both are missing.

In [20]:
# ---7.2.Remove redundant email features--------------------------------------

# provider match duplicates domain match signal very closely
# we keep the domain match feature and drop provider match

dev_df.drop(columns=["email_provider_match"], inplace=True)

print("Dropped redundant feature: email_provider_match")

Dropped redundant feature: email_provider_match


# 8.Save modeling dataset

In [22]:
# ---8.1.Remove columns excluded from modeling--------------------------------

# TransactionID is a pure identifier
# TransactionDT was used for feature engineering but should not enter the model directly

model_exclude_cols = [
    "TransactionID",
    "TransactionDT"
]

model_df = dev_df.drop(columns=model_exclude_cols)

print("Columns removed from modeling dataset:")
print(model_exclude_cols)


# ---8.2.Create processed data directory--------------------------------------

from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", processed_path.resolve())


# ---8.3.Save development modeling dataset------------------------------------

dev_output_path = processed_path / "dev_modeling_dataset.parquet"

model_df.to_parquet(dev_output_path, index=False)

print("Development modeling dataset saved to:")
print(dev_output_path.resolve())


# ---8.4.Save holdout transaction dataset-------------------------------------

# Holdout remains untouched except for the target split done earlier
# Feature engineering will be applied later using the same pipeline

try:

    holdout_output_path = processed_path / "holdout_transactions.parquet"

    test_transaction_holdout.to_parquet(holdout_output_path, index=False)

    print("Holdout dataset saved to:")
    print(holdout_output_path.resolve())

except NameError:
    
    print("Holdout dataset variable not found in this notebook session.")
    print("Assuming it was already saved during the data inspection stage.")


# ---8.5.Basic dataset summary-------------------------------------------------

print("\nFinal modeling dataset shape:", model_df.shape)

print("\nTarget distribution:")
print(model_df["isFraud"].value_counts(normalize=True))

Columns removed from modeling dataset:
['TransactionID', 'TransactionDT']
Processed data directory: C:\Users\zeina\data\processed
Development modeling dataset saved to:
C:\Users\zeina\data\processed\dev_modeling_dataset.parquet
Holdout dataset variable not found in this notebook session.
Assuming it was already saved during the data inspection stage.

Final modeling dataset shape: (472432, 464)

Target distribution:
isFraud
0    0.964865
1    0.035135
Name: proportion, dtype: float64


### Notebook summary

This notebook implemented the first stage of feature engineering for the fraud risk scoring project.

Key feature families created:

- Transaction amount transformations
- Time-based behavioral features
- Missingness indicators for high-signal attributes
- Email domain structural features

The resulting development modeling dataset was saved for downstream modeling experiments. Raw identifier and raw time index columns were excluded from direct modeling.